# LifeGuard AI — Module 1: Scam Shield (EDA, Leakage-Safe Modeling & Error Analysis)

This notebook walks through the data audit, leakage prevention checks, cross-validation comparison, dual-threshold tuning (`safe` / `suspicious` / `scam`), and untouched test-set error analysis for **Scam Shield**.

In [ ]:
import json
from pathlib import Path
import pandas as pd

REPO_ROOT = Path("../../..").resolve()
META_PATH = REPO_ROOT / "ml" / "scam" / "artifacts" / "metadata.json"

entries = json.loads(META_PATH.read_text(encoding="utf-8"))
print(f"Loaded {len(entries)} trained Scam Shield model reports.")

## 1. Dataset Audit & Leakage Prevention

- **SMS Spam Collection**: Exact duplicate messages are dropped *before* the 80/20 stratified train/test split so no identical text appears in both train and test.
- **PhiUSIIL Phishing URL Dataset**: Note that `label == 0` denotes **phishing** (`target = 1`) and `label == 1` denotes **legitimate** (`target = 0`). Exact duplicate URLs are removed and splits are grouped by normalized hostname (`StratifiedGroupKFold`) so no domain leaks across CV folds or into the test set.
- **Webpage HTML columns are excluded**: At inference time, the user only submits a raw URL string (never fetched over the network), so only lexical URL features from `ml.scam.features.URLLexicalFeatureExtractor` are used inside the `Pipeline`.

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TRACK ===")
    print("Dataset summary:", entry["dataset"])
    print("Split sizes:", entry["splits"])
    print("Tuned thresholds:", entry["thresholds"])
    display(pd.DataFrame(entry["comparison_table"]))

## 2. Final Untouched Test Evaluation & Error Analysis

Because missing a real scam/phishing attempt costs more than flagging a borderline message as `suspicious`, `threshold_high` is tuned via **F2-score** (weighing recall 2x precision) on out-of-fold validation predictions, while `threshold_low` flags borderline cases as `suspicious`.

In [ ]:
for entry in entries:
    print(f"=== {entry['sub_model'].upper()} TEST PERFORMANCE ({entry['model_name']}) ===")
    print("Default 0.50:", entry["metrics"]["test_at_0_50"])
    print("Tuned t_high:", entry["metrics"]["test_at_tuned_scam"])
    print("Alert recall (suspicious + scam):", entry["metrics"]["test_alert_recall_including_suspicious"])
    print("Confusion matrix @ t_high:", entry["error_analysis"]["confusion_matrix_at_t_high"])
    print("Sample false negatives:", entry["error_analysis"]["false_negatives_sample"][:3])
    print("Sample false positives:", entry["error_analysis"]["false_positives_sample"][:3])
    print("Limitations:")
    for lim in entry["limitations"]:
        print(" -", lim)
    print()